In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import datamol as dm

In [2]:
df = pd.read_csv("../data/Vapor_pressures.csv")
df.head()

,Name,CAS-RN,Formula,SMILES,T,Pvap,KDB_ID,Class,Subclass
0,METHANE,74-82-8,CH4,C,90.670000,11.702099,1,Paraffinic Hydrocarbons,n-Alkanes
1,METHANE,74-82-8,CH4,C,94.115172,17.886999,1,Paraffinic Hydrocarbons,n-Alkanes
2,METHANE,74-82-8,CH4,C,97.560345,26.489535,1,Paraffinic Hydrocarbons,n-Alkanes
3,METHANE,74-82-8,CH4,C,101.005517,38.137051,1,Paraffinic Hydrocarbons,n-Alkanes
4,METHANE,74-82-8,CH4,C,104.450690,53.534190,1,Paraffinic Hydrocarbons,n-Alkanes


In [3]:
def get_fp(mol, fp_type='topological'):
    return dm.to_fp(mol, fp_type=fp_type)

df['Mol'] = df['SMILES'].apply(dm.to_mol)
df['Fingerprint'] = df['Mol'].apply(get_fp)
df.head()

[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not removing hydrogen atom without neighbors
[11:57:18] WARNING: not r

,Name,CAS-RN,Formula,SMILES,T,Pvap,KDB_ID,Class,Subclass,Mol,Fingerprint
0,METHANE,74-82-8,CH4,C,90.670000,11.702099,1,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x0000026840E...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,METHANE,74-82-8,CH4,C,94.115172,17.886999,1,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x0000026840E...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,METHANE,74-82-8,CH4,C,97.560345,26.489535,1,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x0000026840E...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,METHANE,74-82-8,CH4,C,101.005517,38.137051,1,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x0000026840E...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,METHANE,74-82-8,CH4,C,104.450690,53.534190,1,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x0000026840E...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [4]:
df["ln_P"] = np.log(df.Pvap)


In [5]:
df.columns.tolist()

['Name',
 'CAS-RN',
 'Formula',
 'SMILES',
 'T',
 'Pvap',
 'KDB_ID',
 'Class',
 'Subclass',
 'Mol',
 'Fingerprint',
 'ln_P']

In [6]:
# Splitting the dataset in such a way that the training data/compounds should not be in test data/compounds

from sklearn.model_selection import train_test_split
chemical_classes = df["Class"].unique()
# 3 different dataset for storing the names of the compounds
train_comps, valid_comps, test_comps = [], [], []
for cl in chemical_classes:
    # take all the data belonging to cl class
    df_cl = df[df["Class"]==cl]
    # Get unique compound for this data belonging to cl class
    unique_compound_cl = df_cl["Name"].unique().tolist()
    # Split into train and test
    train_all, comp_test = train_test_split(unique_compound_cl, test_size = 0.20, random_state = 0)
    # Split the train all to train and valid 
    comp_train, comp_valid = train_test_split(train_all, test_size = 0.20, random_state = 1)

    train_comps.extend(list(comp_train))
    valid_comps.extend(list(comp_valid))
    test_comps.extend(list(comp_test))

In [7]:
print(train_comps)

['4-METHYLHEPTANE', 'ISOBUTANE', '3,3-DIMETHYLHEPTANE', '2,2-DIMETHYLOCTANE', '3-METHYLHEXANE', 'N-HEXADECANE', 'N-BUTANE', '3,4-DIMETHYLHEPTANE', '3-METHYLNONANE', '3,3,4-TRIMETHYLHEPTANE', '4-METHYLNONANE', 'N-HEXATRIACONTANE', 'N-DODECANE', '2,2,6-TRIMETHYLHEPTANE', '3-METHYL-3-ETHYLPENTANE', '2,4-DIMETHYL-3-ETHYLHEXANE', '2,4-DIMETHYLHEPTANE', '3-ETHYLHEXANE', '4-ETHYLOCTANE', '2,4,5-TRIMETHYLHEPTANE', '2,4-DIMETHYL-3-ISOPROPYLPENTANE', '2,3,3-TRIMETHYLHEPTANE', '2,3,4,4-TETRAMETHYLHEXANE', '2-METHYL-4-ETHYLHEPTANE', '2,3,3,5-TETRAMETHYLHEXANE', '3,3,5-TRIMETHYLHEPTANE', '2,5-DIMETHYLHEXANE', '2,2,4-TRIMETHYLHEXANE', 'N-PENTADECANE', '2,3,6-TRIMETHYLHEPTANE', 'N-EICOSANE', '2,2,4-TRIMETHYLPENTANE', '2,2,3,3-TETRAMETHYLBUTANE', '2-METHYLHEXANE', 'N-UNDECANE', 'N-TRICOSANE', 'N-TRITRIACONTANE', '3-METHYLHEPTANE', '2,3,4-TRIMETHYLHEPTANE', '2,3,5-TRIMETHYLHEXANE', '2,2,3,4,4-PENTAMETHYLPENTANE', '3,3-DIMETHYL-4-ETHYLHEXANE', '4,5-DIMETHYLOCTANE', 'N-TRIDECANE', '2,3-DIMETHYL-4-ETHYLHE

In [8]:
df_train = df[df["Name"].isin(train_comps)].reset_index()
df_valid = df[df["Name"].isin(valid_comps)].reset_index()
df_test = df[df["Name"].isin(test_comps)].reset_index()

In [9]:
df_train.head()

,index,Name,CAS-RN,Formula,SMILES,T,Pvap,KDB_ID,Class,Subclass,Mol,Fingerprint,ln_P
0,90,N-BUTANE,106-97-8,C4H10,CCCC,134.860000,0.000669,4,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x0000026840E...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-7.309849
1,91,N-BUTANE,106-97-8,C4H10,CCCC,144.871035,0.003877,4,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x0000026840E...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-5.552663
2,92,N-BUTANE,106-97-8,C4H10,CCCC,154.882069,0.017450,4,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x0000026840E...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-4.048429
3,93,N-BUTANE,106-97-8,C4H10,CCCC,164.893103,0.064001,4,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x0000026840E...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-2.748860
4,94,N-BUTANE,106-97-8,C4H10,CCCC,174.904138,0.198505,4,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x0000026840E...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-1.616940


In [10]:
print(df_train.shape)
print(df_valid.shape)
print(df_test.shape)

(26310, 13)
(6810, 13)
(8610, 13)


In [11]:
df_train.columns.tolist()

['index',
 'Name',
 'CAS-RN',
 'Formula',
 'SMILES',
 'T',
 'Pvap',
 'KDB_ID',
 'Class',
 'Subclass',
 'Mol',
 'Fingerprint',
 'ln_P']

In [12]:
# Standardized the temp. values
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(df_train['T'].to_numpy().reshape(-1,1))

df_train['T_scaled'] = scaler.transform(df_train['T'].to_numpy().reshape(-1,1))
df_valid['T_scaled'] = scaler.transform(df_valid['T'].to_numpy().reshape(-1,1))
df_test['T_scaled'] = scaler.transform(df_test['T'].to_numpy().reshape(-1,1))

In [13]:
df["Fingerprint"].tolist()

[array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0,

In [14]:
def getX_y(A,):
    X = np.vstack(A["Fingerprint"].tolist())
    T = A["T_scaled"].to_numpy().reshape(-1,1)
    X = np.hstack((X, T))
    y = A['ln_P'].to_numpy().reshape(-1,1)
    return X, y

In [15]:
X_valid, y_valid = getX_y(df_valid)
X_test,  y_test  = getX_y(df_test)
X_train, y_train = getX_y(df_train)

In [16]:
X_train.shape, y_train.shape

((26310, 2049), (26310, 1))

In [17]:
X_test.shape, X_valid.shape, y_test.shape, y_valid.shape

((8610, 2049), (6810, 2049), (8610, 1), (6810, 1))

In [18]:
# Let's start building the DecisionTreeRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score as R2

model_Tree = DecisionTreeRegressor(random_state=42)
model_Tree.fit(X_train, y_train.ravel())
y_train_Tree = model_Tree.predict(X_train)
print("For training data: ")
print("Train MAE:", MAE(y_train, y_train_Tree))
print("MAPE: ", MAPE(y_train, y_train_Tree))
print("MAE: ", MAE(y_train, y_train_Tree))
print("R2: ", R2(y_train, y_train_Tree))
print("RMSE: ", root_mean_squared_error(y_train, y_train_Tree))
y_valid_Tree = model_Tree.predict(X_valid)
print("For validation data: ")
print("MAPE: ", MAPE(y_valid, y_valid_Tree))
print("MAE: ", MAE(y_valid, y_valid_Tree))
print("R2: ", R2(y_valid, y_valid_Tree))
print("RMSE: ", root_mean_squared_error(y_valid, y_valid_Tree))


For training data: 
Train MAE: 0.001971611908194476
MAPE:  0.001742076828771046
MAE:  0.001971611908194476
R2:  0.9997264430769087
RMSE:  0.07060296178486072
For validation data: 
MAPE:  1.7741155063597316
MAE:  1.7445972673990235
R2:  0.5532660763556234
RMSE:  2.74224839656232


In [19]:
# Make a function for calling the model and fitting it 
import time
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score as R2

def fit_predict(model, X_train, y_train, X_valid):
    start = time.time()
    model.fit(X_train, y_train.ravel())
    fit_time = time.time() - start
    return model, model.predict(X_train), model.predict(X_valid), fit_time

def model_metrics(name, y_train, y_train_pred, y_valid, y_valid_pred, fit_time=None):
    y_train, y_train_pred = np.ravel(y_train), np.ravel(y_train_pred)
    y_valid, y_valid_pred = np.ravel(y_valid), np.ravel(y_valid_pred)
    return {
        "model": name,
        "train_MAE": MAE(y_train, y_train_pred),
        "train_RMSE": root_mean_squared_error(y_train, y_train_pred),
        "train_R2": R2(y_train, y_train_pred),
        "valid_MAE": MAE(y_valid, y_valid_pred),
        "valid_RMSE": root_mean_squared_error(y_valid, y_valid_pred),
        "valid_R2": R2(y_valid, y_valid_pred),
        "fit_time_s": fit_time,
    }

In [20]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

models_to_try = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1),
}

results = []
for name, mdl in models_to_try.items():
    fitted, p_tr, p_va, t = fit_predict(mdl, X_train, y_train, X_valid)
    results.append(model_metrics(name, y_train, p_tr, y_valid, p_va, t))

pd.DataFrame(results)

,model,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,fit_time_s
0,Decision Tree,0.001972,0.070603,0.999726,1.744597,2.742248,0.553266,8.099294
1,Random Forest,0.314885,0.582203,0.981398,1.437179,2.254949,0.697929,189.915517
2,XGBoost,0.783024,1.339202,0.901578,1.323195,2.110358,0.735426,4.469227


In [ ]:
# Compare with the Neural Network
import json
import torch
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # cwd is notebook/, its parent is DNN_VLE/

# 1. the DNN class must be defined in this notebook (or imported) before this point
from src.model import DNN
# 2. read the settings and rebuild the same architecture
with open("models/config.json") as f:
    cfg = json.load(f)

assert X_train.shape[1] == cfg["in_dim"], "input width differs from the saved model"

dnn = DNN(cfg["in_dim"], cfg["hidden_dim"], cfg["out_dim"])

# 3. load the saved weights and switch to evaluation mode
dnn.load_state_dict(torch.load("models/dnn_vle.pt"))
dnn.eval()

# 4. prediction function for the network
def predict_dnn(model, X):
    with torch.no_grad():
        return model(torch.from_numpy(X).float()).numpy().ravel()

p_train = predict_dnn(dnn, X_train)
p_valid = predict_dnn(dnn, X_valid)

# 5. same metric function as the tree models
dnn_row = model_metrics("DNN (size 50)", y_train, p_train, y_valid, p_valid)
results.append(dnn_row)
pd.DataFrame(results)

,model,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,fit_time_s
0,Decision Tree,0.001972,0.070603,0.999726,1.744597,2.742248,0.553266,8.099294
1,Random Forest,0.314885,0.582203,0.981398,1.437179,2.254949,0.697929,189.915517
2,XGBoost,0.783024,1.339202,0.901578,1.323195,2.110358,0.735426,4.469227
3,DNN (size 50),0.740533,1.360149,0.898475,1.712894,2.491605,0.631198,NaN


In [22]:
# Hyperparamter tuning og XGBoost

import numpy as np
import pandas as pd
from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from xgboost import XGBRegressor

# one group label per training row: the compound name
groups_train = df_train['Name'].to_numpy()
assert len(groups_train) == len(X_train), "groups and X_train must have the same length"

# parameter ranges (sampled randomly, not a full grid)
param_distributions = {
    "n_estimators":     randint(200, 1001),        # 200 to 1000 trees
    "learning_rate":    loguniform(0.02, 0.3),     # log scale, includes the 0.3 default
    "max_depth":        randint(3, 9),             # 3 to 8
    "min_child_weight": randint(1, 11),            # 1 to 10
    "subsample":        uniform(0.6, 0.4),         # 0.6 to 1.0
    "colsample_bytree": uniform(0.3, 0.7),         # 0.3 to 1.0
    "reg_lambda":       loguniform(0.1, 10),       # L2 penalty
}

cv = GroupKFold(n_splits=5)

search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=30,                                     # 30 sampled combinations
    scoring={"MAE": "neg_mean_absolute_error",
             "RMSE": "neg_root_mean_squared_error"},
    refit="MAE",                                   # pick the winner by MAE
    cv=cv,
    random_state=42,
    verbose=2,                                     # prints each fit as it runs
    return_train_score=True,
)

search.fit(X_train, y_train.ravel(), groups=groups_train)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[CV] END colsample_bytree=0.5621780831931538, learning_rate=0.262516606324193, max_depth=5, min_child_weight=8, n_estimators=900, reg_lambda=1.5620693675639872, subsample=0.7783331011414365; total time=  23.3s
[CV] END colsample_bytree=0.5621780831931538, learning_rate=0.262516606324193, max_depth=5, min_child_weight=8, n_estimators=900, reg_lambda=1.5620693675639872, subsample=0.7783331011414365; total time=  23.4s
[CV] END colsample_bytree=0.5621780831931538, learning_rate=0.262516606324193, max_depth=5, min_child_weight=8, n_estimators=900, reg_lambda=1.5620693675639872, subsample=0.7783331011414365; total time=  24.2s
[CV] END colsample_bytree=0.5621780831931538, learning_rate=0.262516606324193, max_depth=5, min_child_weight=8, n_estimators=900, reg_lambda=1.5620693675639872, subsample=0.7783331011414365; total time=  23.9s
[CV] END colsample_bytree=0.5621780831931538, learning_rate=0.262516606324193, max_depth=5, min_ch

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': <scipy.stats....002680F9EC110>, 'learning_rate': <scipy.stats....002680F863A40>, 'max_depth': <scipy.stats....002680F27FBC0>, 'min_child_weight': <scipy.stats....002680FAA7FE0>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","{'MAE': 'neg_mean_absolute_error', 'RMSE': 'neg_root_mean_squared_error'}"
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'MAE'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.2

In [23]:
print("Best parameters:", search.best_params_)
print("Best CV MAE:", -search.best_score_)

cv_results = pd.DataFrame(search.cv_results_)
cols = ["rank_test_MAE", "mean_test_MAE", "std_test_MAE", "mean_test_RMSE",
        "mean_train_MAE", "mean_fit_time"] + [c for c in cv_results if c.startswith("param_")]
cv_results[cols].sort_values("rank_test_MAE").head(10)

Best parameters: {'colsample_bytree': np.float64(0.9484106149593443), 'learning_rate': np.float64(0.11661542364152026), 'max_depth': 6, 'min_child_weight': 7, 'n_estimators': 400, 'reg_lambda': np.float64(1.764396768338155), 'subsample': np.float64(0.7323592099410596)}
Best CV MAE: 1.2981244195502908


,rank_test_MAE,mean_test_MAE,std_test_MAE,mean_test_RMSE,mean_train_MAE,mean_fit_time,param_colsample_bytree,param_learning_rate,param_max_depth,param_min_child_weight,param_n_estimators,param_reg_lambda,param_subsample
13,1,-1.298124,0.046908,-2.080942,-0.653758,12.017308,0.948411,0.116615,6,7,400,1.764397,0.732359
17,2,-1.301128,0.058527,-2.074248,-0.614243,17.413688,0.632759,0.091938,6,5,606,1.617465,0.815936
15,3,-1.302407,0.066277,-2.084807,-0.678948,33.087841,0.630550,0.027649,8,3,968,2.776253,0.694394
20,4,-1.302792,0.052469,-2.074372,-0.645725,21.472688,0.385462,0.052489,8,1,584,0.285674,0.770843
1,5,-1.306778,0.041755,-2.073264,-0.608298,26.231920,0.369982,0.069366,7,4,863,2.003443,0.622565
14,6,-1.306980,0.057096,-2.083363,-0.674300,26.728793,0.344491,0.046427,7,3,883,1.884148,0.954885
2,7,-1.306986,0.051473,-2.080326,-0.662073,10.465355,0.805399,0.254012,4,5,360,0.405961,0.809903
4,8,-1.313266,0.052349,-2.080503,-0.724418,13.566501,0.363425,0.106735,6,7,443,1.530485,0.618580
28,9,-1.314626,0.058924,-2.090330,-0.768409,17.147319,0.900851,0.048349,6,3,545,1.057271,0.690598
22,10,-1.319956,0.060991,-2.093777,-0.781396,22.779778,0.383906,0.049900,5,6,871,1.090388,0.881208


In [24]:
# Testing on the validation set
best_xgb = search.best_estimator_
p_tr = best_xgb.predict(X_train)
p_va = best_xgb.predict(X_valid)
results.append(model_metrics("XGBoost (tuned)", y_train, p_tr, y_valid, p_va))
pd.DataFrame(results)

,model,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,fit_time_s
0,Decision Tree,0.001972,0.070603,0.999726,1.744597,2.742248,0.553266,8.099294
1,Random Forest,0.314885,0.582203,0.981398,1.437179,2.254949,0.697929,189.915517
2,XGBoost,0.783024,1.339202,0.901578,1.323195,2.110358,0.735426,4.469227
3,DNN (size 50),0.740533,1.360149,0.898475,1.712894,2.491605,0.631198,NaN
4,XGBoost (tuned),0.686056,1.245895,0.914815,1.272552,2.048637,0.750675,NaN


In [25]:
# save the tuned model
import json
from pathlib import Path
from xgboost import XGBRegressor

Path("models").mkdir(exist_ok=True)

best_xgb = search.best_estimator_

# 1. the fitted model, in XGBoost's own format
best_xgb.save_model("models/xgb_tuned.json")

# 2. the tuned parameters and what is needed to use the model
best_params = {k: (int(v) if isinstance(v, (int, np.integer)) else float(v))
               for k, v in search.best_params_.items()}      # JSON can't store NumPy numbers

cfg_xgb = {
    "model": "XGBRegressor",
    "tuned_params": best_params,
    "fixed_params": {"random_state": 42},
    "search": "RandomizedSearchCV, 30 iterations, GroupKFold(5) by compound name, scored by MAE",
    "cv_MAE": float(-search.best_score_),
    "input": "2048-bit topological fingerprint + temperature scaled with models/temperature_scaler.joblib",
    "target": "ln(P_vap), P in kPa",
    "valid_MAE": 1.272552,
    "valid_RMSE": 2.048637,
    "valid_R2": 0.750675,
}
with open("models/xgb_config.json", "w") as f:
    json.dump(cfg_xgb, f, indent=2)

In [26]:
# Checking
loaded_xgb = XGBRegressor()
loaded_xgb.load_model("models/xgb_tuned.json")

print(np.allclose(best_xgb.predict(X_valid), loaded_xgb.predict(X_valid)))   # should print True

True


In [27]:
# testin on the test set
def model_metrics_test(name, y_test, y_test_pred ):
    y_test, y_test_pred = np.ravel(y_test), np.ravel(y_test_pred)
    return {
        "model": name,
        "test_MAE": MAE(y_test, y_test_pred),
        "test_RMSE": root_mean_squared_error(y_test, y_test_pred),
        "test_R2": R2(y_test, y_test_pred),
    }
    

In [29]:
models_to_try = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1),
}

results_test = []
for name, mdl in models_to_try.items():
    mdl.fit(X_train, y_train.ravel())
    y_test_pred = mdl.predict(X_test)
    results_test.append(model_metrics_test(name, y_test, y_test_pred ))

pd.DataFrame(results_test)

,model,test_MAE,test_RMSE,test_R2
0,Decision Tree,1.649637,2.706757,0.586113
1,Random Forest,1.353214,2.264492,0.710316
2,XGBoost,1.270246,2.008173,0.772183


In [31]:
results_test.append(model_metrics_test("XGBoost (tuned)", y_test, best_xgb.predict(X_test) ))
pd.DataFrame(results_test)

,model,test_MAE,test_RMSE,test_R2
0,Decision Tree,1.649637,2.706757,0.586113
1,Random Forest,1.353214,2.264492,0.710316
2,XGBoost,1.270246,2.008173,0.772183
3,XGBoost (tuned),1.230471,1.967815,0.781248
4,XGBoost (tuned),1.230471,1.967815,0.781248


In [32]:
results_test.pop()

{'model': 'XGBoost (tuned)',
 'test_MAE': 1.230471497162451,
 'test_RMSE': 1.9678145464244359,
 'test_R2': 0.7812481427920833}

In [33]:
pd.DataFrame(results_test)

,model,test_MAE,test_RMSE,test_R2
0,Decision Tree,1.649637,2.706757,0.586113
1,Random Forest,1.353214,2.264492,0.710316
2,XGBoost,1.270246,2.008173,0.772183
3,XGBoost (tuned),1.230471,1.967815,0.781248


In [34]:
y_test_dnn = predict_dnn(dnn, X_test)

# 5. same metric function as the tree models
dnn_row = model_metrics_test("DNN (size 50)", y_test, y_test_dnn)
results_test.append(dnn_row)
pd.DataFrame(results_test)

,model,test_MAE,test_RMSE,test_R2
0,Decision Tree,1.649637,2.706757,0.586113
1,Random Forest,1.353214,2.264492,0.710316
2,XGBoost,1.270246,2.008173,0.772183
3,XGBoost (tuned),1.230471,1.967815,0.781248
4,DNN (size 50),1.634303,2.516407,0.642278
